In [1]:
# load required libraries
import sys
sys.path.append('../utils')
# data model - Reading annotated data (by annotators)
# the data model class below returns masks as well
from json_parser import CellMaskDataset
import os
from PIL import Image
from IPython.display import display
import numpy as np
import cv2
import pandas as pd
from typing import List, Union, Dict, Final, Tuple, Optional

In [ ]:
def get_num_overlapping_objects(boxes: Union[List[Tuple], np.ndarray], masks: Union[None, List[np.ndarray]]) \
-> Tuple[List[float], List[float]]:
    """
    Given a set of bounding boxes and mask within each box, this function first constructs 
    the smallest box (rectangle) covering the union of these boxes. Then for each pixel in
    this box, it calculates the number of individual object masks covering
    that pixel. 
    Args:
        boxes (list of 4-tuples or lists with 4 integer elements or a N by 4 numpy array): Each element/row
            is a bounding box (x1, y1, x2, y2) where (x1, y1) is the top-left and (x2, y2) is the bottom-right 
            corner of the bounding box.
        masks (list of numpy arrays): Each element is a numpy array of size (x2 - x1, y2 - y1) defining the object
            mask within the provided bounding box above. 
    Returns:
        A list with the same size as the inputs boxes and masks, wich each element indicating the number of
        overlapping objects. 
        A list with the same size as the inputs boxes and masks, wich each element indicating the average number of
        overlapping objects. 
    """
    if len(boxes) == 0:
        return [], []

    if isinstance(boxes, list):
        boxes_array: np.array(boxes)
    else:
        boxes_array = boxes

    min_x: int = np.min(boxes_array[:, 0])
    min_y: int = np.min(boxes_array[:, 1])
    max_x: int = np.max(boxes_array[:, 2])
    max_y: int = np.max(boxes_array[:, 3])
    
    object_count_per_pixel = np.zeros((max_y - min_y, max_x - min_x), dtype=np.uint8)

    for i, box in enumerate(boxes):
        (x1, y1, x2, y2) = box
        if masks is not None:
            object_count_per_pixel[y1 - min_y:y2 - min_y, x1 - min_x:x2 - min_x] += masks[i]
        else:
            object_count_per_pixel[y1 - min_y:y2 - min_y, x1 - min_x:x2 - min_x] += 1

    num_overlapping_objects: List[int] = []
    mean_overlapping_objects: List[float] = []
    for i, box in enumerate(boxes):
        (x1, y1, x2, y2) = box
        if masks is not None:
            overlapping_objects_per_pixel = object_count_per_pixel[y1 - min_y:y2 - min_y, x1 - min_x:x2 - min_x] * masks[i]
        else:
            overlapping_objects_per_pixel = object_count_per_pixel[y1 - min_y:y2 - min_y, x1 - min_x:x2 - min_x]
        num_overlapping_objects.append(np.max(overlapping_objects_per_pixel) - 1)
        mean_overlapping_objects.append(np.mean(overlapping_objects_per_pixel[overlapping_objects_per_pixel > 0]) - 1)

    return num_overlapping_objects, mean_overlapping_objects

In [ ]:
SUSPENSION_CLASS_IDS: List[int] = [1]
ADHERED_CLASS_IDS: List[int] = [2, 3]
ANNOTATIONS_CLASS_NAMES_TO_CLASS_IDS_MAP: Dict[str, int] = {
    'cell': SUSPENSION_CLASS_IDS[0], 
    'dead-cell': SUSPENSION_CLASS_IDS[0], 
    'cytoplasm': ADHERED_CLASS_IDS[0],
    'cell-adhered': ADHERED_CLASS_IDS[0], 
    'soma': ADHERED_CLASS_IDS[1], 
    # 'cage': 4, # including cages will result in excluding caged cells because of overlap
    # 'cages': 4
    'bead': 5
}

CONSIDER_ADHERED_CELLS: bool = True
USE_MASKS_FOR_OVERLAP_CALC: bool = True
KEEP_OBJECT_SIZES: bool = False

ONLY_USE_BEST_FOCUS_IMAGE: bool = True
MAX_IMAGE_SIDE: int = 4512
CROP_SIZE: int = 96
PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES: float = 0.15
OVERLAP_THRESHOLD: float = 0.05

if not USE_MASKS_FOR_OVERLAP_CALC:
    # use a higher overlap threshold if masks are not used for calculating overlaps (boxes are used)
    OVERLAP_THRESHOLD: float = 0.1
    
if CONSIDER_ADHERED_CELLS:
    DATASET_PATHS: Dict[str, List[str]] = {
        'IMR90':  ['/media/cellareye/SSD/Data/Cellanome/Segmentation/231212_imr90_multichannel_overlay',
                   '/media/cellareye/SSD/Data/Cellanome/Segmentation/240213_imr90_multichannel_overlay'],
        'Hs675T': ['/home/cellareye/Cellanome/Data/20240509_Hs675Tfibroblasts_10x_caged'], 
        'HeLa':   ['/home/cellareye/Cellanome/Data/20240509_hela-adhered_10x_caged'],
        'MC38':   ['/home/cellareye/Cellanome/Data/20240624_mc38_10x_caged',
                   '/home/cellareye/Cellanome/Data/20240624_mc38_10x_uncaged',
                   '/home/cellareye/Cellanome/Data/20240625_mc38_10x_caged'],
        'mutuDC': ['/home/cellareye/Cellanome/Data/20240515_DC-adhered_10x_caged', 
                   '/home/cellareye/Cellanome/Data/20240516_DC-adhered_10x_caged'],
        'Neuron': ['/home/cellareye/Cellanome/Data/20240422_neuron-adhered_10x_uncaged', 
                   '/home/cellareye/Cellanome/Data/20240703_neuron-adhered_10x_caged', 
                   '/home/cellareye/Cellanome/Data/20240704_neuron-adhered_10x_caged']
    }
else:
    DATASET_PATHS: Dict[str, List[str]] = {
        'Jurkat':  ['/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_jurkat_10x_caged',
                    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_jurkat_10x_uncaged'],
        'K562':    ['/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_k562_10x_caged',
                    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_k562_10x_uncaged'],
        'Raji':    ['/home/cellareye/Cellanome/Data/20240905_raji_10x_caged_at_4x'],
        'HeLa':    ['/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_hela-suspension_10x_caged',
                    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_hela-suspension_10x_uncaged'],
        'NK92':    ['/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_nk92_10x_caged',
                    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240228_nk92_10x_uncaged',
                    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240425_nk92_10x_caged',
                    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240425_nk92_10x_uncaged'],
        'IMR90':   ['/media/cellareye/SSD/Data/Cellanome/Segmentation/20240314_imr90-suspension_10x_caged',
                    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240314_imr90-suspension_10x_uncaged'],
        'PBMC':    ['/media/cellareye/SSD/Data/Cellanome/Segmentation/20240307_pbmc-beads_10x_uncaged',
                    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240305_pbmc-nobeads_10x_caged',
                    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240305_pbmc-nobeads_10x_uncaged',
                    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240306_mousepbmc-beads_10x_caged',
                    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240306_mousepbmc-beads_10x_uncaged',
                    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240306_mousepbmc-nobeads_10x_caged',
                    '/media/cellareye/SSD/Data/Cellanome/Segmentation/20240306_mousepbmc-nobeads_10x_uncaged'],
        'TALL104': ['/home/cellareye/Cellanome/Data/20240816_tall104_10x_caged_at_4x'], 
    }

CLASSIFIER_LABEL_MAP = {k: v for k, v in enumerate(DATASET_PATHS.keys())}

if CONSIDER_ADHERED_CELLS:
    cell_state_str: str = 'adhered'
else:
    cell_state_str: str = 'suspension'

if USE_MASKS_FOR_OVERLAP_CALC:
    OUTPUT_FOLDER = 'cellanome_' + cell_state_str + '_cells_classification_data_mask'
else:
    OUTPUT_FOLDER = 'cellanome_' + cell_state_str + '_cells_classification_data_box'

if not os.path.exists(OUTPUT_FOLDER):
    os.mkdir(OUTPUT_FOLDER)

print(f"Classifier label map: {CLASSIFIER_LABEL_MAP}")

## Create cropped images of Cellanome cells
Skip this step onces the training images are created. This step should be repeated for any additional dataset/cell type available. 

In [ ]:
def create_dataset_classes(dataset_path: str, 
                           class_names_to_class_ids_map: Dict[str, int] = ANNOTATIONS_CLASS_NAMES_TO_CLASS_IDS_MAP, 
                           train: bool = True, 
                           percentage_to_expand_bbox_boundaries: float = PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES, 
                           max_larger_side: int = MAX_IMAGE_SIDE, 
                           max_smaller_side: int = MAX_IMAGE_SIDE):
    
    images_path:str = dataset_path
    annotations_path: str = os.path.join(dataset_path, 'annotations')

    annotations_images_map: pd.DataFrame = pd.read_csv(os.path.join(dataset_path, 'annotation_images_mapping.csv'))

    test_files: List[str] = []
    train_files: List[str] = []
    
    with open(os.path.join(dataset_path, 'test.txt')) as file:
        filenames = file.readlines()
        filenames = [f.replace('\n', '') for f in filenames if len(f) > 0]
        test_files += filenames

    with open(os.path.join(dataset_path, 'train.txt')) as file:
        filenames = file.readlines()
        filenames = [f.replace('\n', '') for f in filenames if len(f) > 0]
        train_files += filenames

 
    columns: List[str] = list(annotations_images_map.columns)
    image_columns = [column_name for column_name in columns if 'white_dz' in column_name.lower()]

    if len(image_columns) > 0 and ONLY_USE_BEST_FOCUS_IMAGE:
        image_columns = [column_name for column_name in columns if 'white_dz0' in column_name.lower()]
        
    if len(image_columns) == 0:
        # this is not a focus sweep dataset, get the BF image
        for column_name in columns:
            if 'bf' in column_name.lower() or 'white' in column_name.lower():
                image_columns = [column_name]
                break

    if len(image_columns) == 0:
        print('[ERROR]: No brightfield image folder could be extracted from annotation_images_mapping.csv file for the dataset')
    
    train_map_dict: Dict[str, List[str]] = {}
    test_map_dict: Dict[str, List[str]] = {}
    for _, row in annotations_images_map.iterrows():
        annotations_filename = row['annotation_json']
        name = '.'.join(annotations_filename.strip().split('.')[:-1])
        if name in test_files:
            test_map_dict[annotations_filename] = list(row[image_columns])
        else:
            train_map_dict[annotations_filename] = list(row[image_columns])


    # datasets
    if train:
        dataset = CellMaskDataset(images_path=images_path, annotations_path=annotations_path, 
                                  annotations=train_map_dict,
                                  max_images_to_consider_for_each_annotation=1, # in case of focus sweep dataset, only pick 1 image randomly
                                  labels_of_interest=list(class_names_to_class_ids_map.keys()), 
                                  percentage_to_expand_bbox_boundaries = percentage_to_expand_bbox_boundaries, 
                                  color_depth=8, 
                                  min_object_diameter = 6.0,
                                  scale_factor_dict={}, 
                                  max_larger_side = max_larger_side, max_smaller_side = max_smaller_side,
                                  normalize=False, class_names_to_ids_map=class_names_to_class_ids_map)

    else:
        dataset = CellMaskDataset(images_path=images_path, annotations_path=annotations_path, 
                                  annotations=test_map_dict,
                                  max_images_to_consider_for_each_annotation=1,
                                  labels_of_interest=list(class_names_to_class_ids_map.keys()), 
                                  percentage_to_expand_bbox_boundaries = percentage_to_expand_bbox_boundaries, 
                                  color_depth=8, 
                                  min_object_diameter = 6.0,
                                  scale_factor_dict={}, 
                                  max_larger_side = max_larger_side, max_smaller_side = max_smaller_side,
                                  normalize=False, class_names_to_ids_map=class_names_to_class_ids_map)

    return dataset

In [ ]:
def create_training_images(dataset_id: int, 
                           dataset: CellMaskDataset, 
                           cell_type: str,
                           classifier_label_map: Dict[int, str],
                           output_base_folder: str, 
                           crop_size: int, 
                           use_masks_for_overlap_calc: bool,
                           train: bool = True, 
                           adhered_state: bool = True,
                           keep_obj_sizes: bool = False,
                           ):

    half_crop_size: int = int(np.ceil(crop_size / 2.0))
    percentage_of_used_objects: List[float] = []
    reverse_label_map = {v: k for k, v in classifier_label_map.items()}
    
    set_type_str: str = "train" if train else "test"
    output_folder: str = os.path.join(output_base_folder, set_type_str)
    if not os.path.exists(output_folder):
        os.mkdir(output_folder)
    
    for idx in range(len(dataset)):
        # read the sample 
        sample: dict = dataset[idx] 
        if len(sample['annotations']) == 0:
            continue
        
        # image
        img: np.ndarray = sample['image']

        img_height, img_width = img.shape[:2]
        
        img_name: str = '.'.join(sample['name'].strip().split('.')[:-1])
        obj_count: int = 0

        # get the average overlapping objects for each annotate object, we use this to pick the samples
        # that have small overlaps with other cells
        obj_boxes: np.ndarray = sample['annotations'][['xtl', 'ytl', 'xbr', 'ybr']].values.astype(int)
        boxes_widths: np.ndarray = obj_boxes[:, 2] - obj_boxes[:, 0]
        boxes_heights: np.ndarray = obj_boxes[:, 3] - obj_boxes[:, 1]
        
        boxes_centers_x: np.ndarray = (obj_boxes[:, 2] + obj_boxes[:, 0]) / 2.0
        boxes_centers_y: np.ndarray = (obj_boxes[:, 3] + obj_boxes[:, 1]) / 2.0
            
        half_obj_sizes: np.ndarray = np.maximum(boxes_heights, boxes_widths) / 2.0
        
        if use_masks_for_overlap_calc:
            obj_masks: List[np.ndarray] = sample['masks']
            obj_areas: List[int] = [np.sum(mask) for mask in obj_masks]
            min_mask_area: int = np.percentile(obj_areas, 20)
        else:
            # expand the bbox to create a square bounding box around the object
            # we do this to avoid having black bands on top/bottom or left/right of the images
            # note that the bounding boxes for objects at the image boundaries may not be sqaure, but we 
            # can ignore those cases for now
            obj_boxes[:, 0] = np.maximum((boxes_centers_x - half_obj_sizes), 0).astype(int)
            obj_boxes[:, 1] = np.maximum((boxes_centers_y - half_obj_sizes), 0).astype(int)
            obj_boxes[:, 2] = np.minimum((boxes_centers_x + half_obj_sizes), img_width).astype(int)
            obj_boxes[:, 3] = np.minimum((boxes_centers_y + half_obj_sizes), img_height).astype(int)
            
            obj_masks = None

        num_overlapping_objects, avg_overlapping_objects = get_num_overlapping_objects(
            obj_boxes, 
            obj_masks)
        
        for obj_id, row in sample['annotations'].iterrows():
            # box coordinates
            xtl, ytl, xbr, ybr = obj_boxes[obj_id]
            label = row['label']

            if adhered_state:
                # we need to only consider cells annotated as an adhered cell
                if label not in ADHERED_CLASS_IDS:
                    continue
            else:
                # we need to only consider cells annotated as a suspension cell
                if label not in SUSPENSION_CLASS_IDS:
                    continue
            
            # check if there are many other overlapping objects for this object
            if avg_overlapping_objects[obj_id] > OVERLAP_THRESHOLD:
                # print(f"This cell from {cell_type} has {num_overlapping_objects[obj_id]} overlapping cells! Skipping ... ")
                continue

            # skip tiny and narrow objects
            if use_masks_for_overlap_calc and np.sum(obj_masks[obj_id]) < min_mask_area:
                continue

            # skip potentially cut and incomplete objects near the image boundary
            if xtl < 5 or ytl < 5 or xbr > img_width - 5 or ybr > img_height - 5:
                continue

            # if we are supposed to keep the object sizes and the object does not fit in the crop, skip it
            if keep_obj_sizes and half_obj_sizes[obj_id] > half_crop_size:
                continue

            # if we are supposed to keep the object sizes and the object is too small compared to the provided crop, skip it
            if keep_obj_sizes and half_obj_sizes[obj_id] < 0.4 * half_crop_size:
                continue
            
            
            # form an sqaure crop around the object and centered around the center of the box
            # note that the bounding box of the object is already expanded by the factor PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES
            # when the dataset is created, so no need to further expand here to have some margin around the object
            start_pixel_in_x: int = int(max(0, boxes_centers_x[obj_id] - half_obj_sizes[obj_id]))
            end_pixel_in_x: int = int(start_pixel_in_x + 2 * half_obj_sizes[obj_id])

            # adjust for the objects on the image boundaries
            if end_pixel_in_x >= img_width:
                end_pixel_in_x = img_width
                start_pixel_in_x = int(img_width - 2 * half_obj_sizes[obj_id])

            start_pixel_in_y: int = int(max(0, boxes_centers_y[obj_id] - half_obj_sizes[obj_id]))
            end_pixel_in_y: int = int(start_pixel_in_y + 2 * half_obj_sizes[obj_id])

            if end_pixel_in_y >= img_height:
                end_pixel_in_y = img_height
                start_pixel_in_y = int(img_height - 2 * half_obj_sizes[obj_id])
            
            # resize the object within the provided crop size, we do that to ensure the crop only 
            # contains the object class as annotated
            if half_obj_sizes[obj_id] > half_crop_size:
                interpolation_scheme = cv2.INTER_AREA
            else:
                interpolation_scheme = cv2.INTER_CUBIC

            # make a copy of the image, and then mask it with dark background
            img_copy = np.zeros(img.shape, np.uint8)
            if use_masks_for_overlap_calc:
                # with this option, xtl, ytl, xbr, ybr have not been extended to for a square crop
                img_copy[ytl:ybr, xtl:xbr] = img[ytl:ybr, xtl:xbr] * sample['masks'][obj_id]
            else:
                # with this option, xtl, ytl, xbr, ybr have been extended to for a square crop
                # they will be eqaul to start_pixel_in_x, start_pixel_in_y, end_pixel_in_x and end_pixel_in_y
                # unless the box is on the boundary of the image
                # in this case, a black band is expected, but we can ignore this small number of objects with issues
                img_copy[ytl:ybr, xtl:xbr] = img[ytl:ybr, xtl:xbr]

            if keep_obj_sizes:
                cropped_img: np.ndarray = np.zeros((crop_size, crop_size), dtype=np.uint8)
                obj_crop: np.ndarray = img_copy[start_pixel_in_y:end_pixel_in_y, start_pixel_in_x:end_pixel_in_x]
                # center the object crop inside the crop
                # if we are here, for sure half_obj_sizes[obj_id] <= half_crop_size
                offset: int = int(half_crop_size - half_obj_sizes[obj_id])
                cropped_img[offset:offset + obj_crop.shape[1], offset:offset + obj_crop.shape[0]] = obj_crop
            else:
                cropped_img: np.ndarray = cv2.resize(img_copy[start_pixel_in_y:end_pixel_in_y, start_pixel_in_x:end_pixel_in_x], 
                                                     (crop_size, crop_size), 
                                                     interpolation_scheme)

            class_id: int = reverse_label_map[cell_type]
            
            output_class_folder: str = os.path.join(output_folder, str(class_id))
            if not os.path.exists(output_class_folder):
                os.mkdir(output_class_folder)
                
            cv2.imwrite(os.path.join(output_class_folder, str(dataset_id) + '_' + img_name + '_' + str(obj_count) + '.jpg'), 
                        cropped_img, [int(cv2.IMWRITE_JPEG_QUALITY), 100])

            obj_count += 1
        
        percentage_of_used_objects.append(float(obj_count) / len(sample['annotations']))
        
    return percentage_of_used_objects

In [ ]:
cell_types_list: List[str] = []
dataset_paths_list: List[str] = []
for cell_type, dataset_paths in DATASET_PATHS.items():
    cell_types_list += [cell_type] * len(dataset_paths)
    dataset_paths_list += dataset_paths

for dataset_id, dataset_path in enumerate(dataset_paths_list):

    dataset_name = os.path.basename(dataset_path)
    
    train_dataset = create_dataset_classes(dataset_path=dataset_path, 
                                           train=True)
    test_dataset = create_dataset_classes(dataset_path=dataset_path, 
                                           train=False)
    
    percentage_of_used_objects = create_training_images(
        dataset_id=dataset_id, 
        dataset=train_dataset, 
        cell_type=cell_types_list[dataset_id],
        classifier_label_map=CLASSIFIER_LABEL_MAP,
        output_base_folder=OUTPUT_FOLDER, 
        crop_size=CROP_SIZE, 
        use_masks_for_overlap_calc=USE_MASKS_FOR_OVERLAP_CALC,
        train=True,
        adhered_state=CONSIDER_ADHERED_CELLS,
        keep_obj_sizes=KEEP_OBJECT_SIZES
        )
    
    print(f"On average used {np.round(np.mean(percentage_of_used_objects) * 100, 2)}% of the objects from train images of {dataset_name}")
    
    percentage_of_used_objects = create_training_images(
        dataset_id=dataset_id, 
        dataset=test_dataset, 
        cell_type=cell_types_list[dataset_id],
        classifier_label_map=CLASSIFIER_LABEL_MAP,
        output_base_folder=OUTPUT_FOLDER, 
        crop_size=CROP_SIZE, 
        use_masks_for_overlap_calc=USE_MASKS_FOR_OVERLAP_CALC,
        train=False,
        adhered_state=CONSIDER_ADHERED_CELLS,
        keep_obj_sizes=KEEP_OBJECT_SIZES
        )
    
    print(f"On average used {np.round(np.mean(percentage_of_used_objects) * 100, 2)}% of the objects from test images of {dataset_name}")

# Training a Classifier

In [ ]:
import torch, torchvision
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import time

import random
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image

## Configurations

In [ ]:
# training batch size
BATCH_SIZE = 64
# learning rate
LEARNING_RATE = 1e-3
# number of training epochs
NUM_EPOCHS = 10
# learning rate decay steps
LR_DECAY_STEPS = 0

## Dataset model - images

In [ ]:
class CellDataset(Dataset):
    """ Dataset of cells of different types (classes) """

    def __init__(self, class_images_dict: Dict[int, List[str]], transform=None):
        """
        Args:
            class_images_dict (dictionary): A dictionary with keys as class 
                IDs (integers 0 to num_classes; 0 reserved for background if exists) and 
                values as the full path to the list of train/test images for the class ID 
                (the name should include the full path to the image).
            transform (callable, optional): Optional transform to be applied
                on a sample
        """
        
        self.image_names: List[str] = []
        self.labels: List[int] = []
        for label, image_paths_list in class_images_dict.items():
            self.image_names = self.image_names + image_paths_list
            self.labels = self.labels + [label] * len(image_paths_list)
        self.transform = transform
        self.num_classes = len(class_images_dict)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        
        image: np.ndarray = cv2.imread(self.image_names[idx], cv2.IMREAD_UNCHANGED)
        
        if np.ndim(image) == 2:
            # for gray scale channel, make them a 3-D image expected by the model
            image = np.repeat(np.expand_dims(image, axis=2), 3, axis=2)
        
        # image_tensor: torch.tensor = torchvision.transforms.functional.to_tensor(image)
        
        if self.transform:
            image = self.transform(image)

        
        return image, self.labels[idx]

### DINOv2 model

In [ ]:
# train and test data transforms
image_transforms = { 
    'train': torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Resize(84), 
        # torchvision.transforms.CenterCrop(84), 
        torchvision.transforms.RandomApply(torch.nn.ModuleList([
            torchvision.transforms.RandomRotation(degrees=[90.0, 90.0])
        ]), p=0.25),
        torchvision.transforms.RandomHorizontalFlip(p=0.5),
        torchvision.transforms.RandomVerticalFlip(p=0.5),
        torchvision.transforms.Normalize([0.485, 0.456, 0.406],
                                         [0.229, 0.224, 0.225])
    ]),
    'test': torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Resize(84), 
        # torchvision.transforms.CenterCrop(84), 
        torchvision.transforms.Normalize([0.485, 0.456, 0.406],
                                         [0.229, 0.224, 0.225])
    ])
}

In [ ]:
class_images_dict_train: Dict[int, List[str]] = {}
class_images_dict_test: Dict[int, List[str]] = {}

if CONSIDER_ADHERED_CELLS:
    cell_state: str = 'Adhered'
else:
    cell_state: str = 'Suspension'

# the number of datasets and the cells for each type varies significantly
# we use this number to equalize the number of samples we use for training and testing
MAX_NUM_TRAIN_SAMPLES: int = 10000
MAX_NUM_TEST_SAMPLES: int = 1000

np.random.seed(7)

for class_id, cell_type in CLASSIFIER_LABEL_MAP.items():
    image_files = os.listdir(os.path.join(OUTPUT_FOLDER, "train" , str(class_id)))
    np.random.shuffle(image_files)
    class_images_dict_train[class_id] = [os.path.join(OUTPUT_FOLDER, "train", str(class_id), f) for f in image_files[:MAX_NUM_TRAIN_SAMPLES]]
    print(f"[INFO] Found {len(class_images_dict_train[class_id])} images of class '{cell_type + '-' + cell_state}' with ID {class_id} for training")  
    image_files = os.listdir(os.path.join(OUTPUT_FOLDER, "test" , str(class_id)))
    np.random.shuffle(image_files)
    class_images_dict_test[class_id] = [os.path.join(OUTPUT_FOLDER, "test", str(class_id), f) for f in image_files[:MAX_NUM_TEST_SAMPLES]]
    print(f"[INFO] Found {len(class_images_dict_test[class_id])} images of class '{cell_type + '-' + cell_state}' with ID {class_id} for testing")  

## Datasets and dataloaders - images

In [ ]:
train_dataset = CellDataset(class_images_dict_train, image_transforms['train'])
train_data_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True)

test_dataset = CellDataset(class_images_dict_test, image_transforms['test'])
test_data_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False)

### Visualize some samples

In [ ]:
def show_sample_batch(sample_batch):
    """ Show training images for a batch """
    images_batch, labels_batch = sample_batch
    batch_size: int = len(labels_batch)
    grid_image = torchvision.utils.make_grid(images_batch, normalize = True)
    plt.imshow(grid_image.numpy().transpose((1, 2, 0)))
    print('Labels:' + ' '.join('%5s' % labels_batch[j].item() for j in range(batch_size)))

In [ ]:
print(CLASSIFIER_LABEL_MAP)

In [ ]:
data_iter = iter(DataLoader(train_dataset, batch_size = 4, shuffle = True))
show_sample_batch(next(data_iter))     

## Model definition
We use a DINOv2 for the classifier using images, after replacing the linear head.

In [ ]:
def build_dinov2_model(num_classes: int, model_type: str = "small", with_registers: bool=False):
    class Dinov2Model(nn.Module):
        def __init__(self, num_classes: int, model_type: str = "small", with_registers: bool=True) -> None:
            super(Dinov2Model, self).__init__()
            # get the pre-trained DINOv2 model based on the passed size for the model
            model_type_map = {"small": "dinov2_vits14", 
                              "base":  "dinov2_vitb14", 
                              "large": "dinov2_vitl14", 
                              "giant": "dinov2_vitg14",
                             }
            if model_type in model_type_map:
                model_type_str: str = model_type_map[model_type]
                
            else:
                model_type_str: str = "dinov2_vitb14"
                print(f"[ERROR] Incorrect model type passed {model_type}! Using the base model by default.")
        
            # DINOv2 with registers
            if with_registers:
                model_type_str += "_reg"
            
            self.dinov2 = torch.hub.load("facebookresearch/dinov2", model_type_str)
            # freeze the model
            for param in self.dinov2.parameters():
                param.requires_grad_(False)

            self.num_classes = num_classes
            # the dimension of the CLS embeddings
            self.num_fc_inputs: int = self.dinov2.norm.normalized_shape[0]
            self.fc = nn.Linear(self.num_fc_inputs, self.num_classes)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.fc(self.dinov2(x))
    
    return Dinov2Model(num_classes=num_classes, model_type=model_type, with_registers=with_registers)

## Training
### Optimizer settings

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print('Device available:' , device)

model = build_dinov2_model(num_classes=112, model_type= "large", with_registers=True)
params = [p for p in model.parameters() if p.requires_grad]

# move model to the right device
model.train()
model.to(device)

# construct an optimizer

optimizer = torch.optim.Adam(params, lr = LEARNING_RATE)
print('Adam Optimizer is configured for %d epochs' %NUM_EPOCHS)

print(f"Initial learning rate is set to {LEARNING_RATE}")
if LR_DECAY_STEPS < 1:
    print(f"One-Cyle LR scheduler is configured for {NUM_EPOCHS} with {len(train_data_loader)} steps per epoch")
    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer, 
                                                       max_lr=LEARNING_RATE, 
                                                       epochs=NUM_EPOCHS,
                                                       steps_per_epoch=len(train_data_loader))
    
else:
    print(f"Step LR scheduler is configured with {LR_DECAY_STEPS} epochs for each step")
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer,
                                                   step_size=LR_DECAY_STEPS,
                                                   gamma=0.1)


criterion = nn.CrossEntropyLoss()

### Training script

In [ ]:
def train(model, 
          train_loader, 
          test_loader, 
          criterion, 
          optimizer, 
          lr_scheduler,
          num_epochs,
          device):
    
    torch.cuda.empty_cache()
    
    # losses, accuracies and mean IoUs over the training epochs
    train_losses: List[float] = []
    test_losses: List[float] = []
    train_accs: List[float] = []
    test_accs: List[float] = []
    test_precision: List[float] = []
    test_recall: List[float] = []
    
    # learning rates used for each step (not each epoch as we may use OneCyle scheduling)
    lrs: List[float] = []
    min_loss: float = np.inf
    
    model = model.to(device)
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        
        since = time.time()
        
        running_loss: float = 0
        accuracy: float = 0
        
        # training loop
        model.train()
        for i, data in enumerate(tqdm(train_loader)):
            # training phase
            image_tensors, label_tensors = data
            
            image_tensors = image_tensors.to(device) 
            label_tensors = label_tensors.to(device)
            
            # forward
            output = model(image_tensors)
            loss = criterion(output, label_tensors)
            # evaluate metrics
            # compute the accuracy
            
            _, predictions = torch.max(output.data, dim=1)
            correct = predictions.eq(label_tensors.data.view_as(predictions)).int()
            
            # convert correct_counts to float and then compute the mean
            accuracy += float(correct.sum()) / float(correct.numel())
            
            # backward
            loss.backward()
            optimizer.step() # update weight          
            optimizer.zero_grad() # reset gradient
            
            # update the learning rate only after one batch in case of One-Cycle LR scheduler
            lrs.append(lr_scheduler.get_last_lr()[0])
            if isinstance(lr_scheduler, torch.optim.lr_scheduler.OneCycleLR):
                lr_scheduler.step() 
            
            running_loss += loss.item()
        
        # update the learning rate after one full epoch if LR step scheduler is used
        if isinstance(lr_scheduler, torch.optim.lr_scheduler.StepLR):
            lr_scheduler.step() 
        
        # run the validation after each training epoch
        model.eval()
        test_running_loss: float = 0
        test_accuracy: float = 0
        
        # validation loop
        with torch.no_grad():
            for i, data in enumerate(tqdm(test_loader)):
                
                image_tensors, label_tensors = data
            
                image_tensors = image_tensors.to(device) 
                label_tensors = label_tensors.to(device)

                output = model(image_tensors)
                # evaluation metrics
                # compute the accuracy
                
                _, predictions = torch.max(output.data, dim=1)
                correct = predictions.eq(label_tensors.data.view_as(predictions)).int()
                
                # convert correct_counts to float and then compute the mean
                test_accuracy += float(correct.sum()) / float(correct.numel())
                # loss
                loss = criterion(output, label_tensors)                                  
                test_running_loss += loss.item()
            
        # calculatio mean for each batch
        running_loss /= len(train_loader)
        accuracy /= len(train_loader)
        
        test_running_loss /= len(test_loader)
        test_accuracy /= len(test_loader)
                       
         # save the results
        train_losses.append(running_loss)
        train_accs.append(accuracy)
        
        test_losses.append(test_running_loss)
        test_accs.append(test_accuracy)
        
        print('saving the model ...')
        torch.save(model.state_dict(), os.path.join('classifier_models', 'checkpoint_' + str(epoch) +'.pt'))
                    
        
        print("Epoch:{}/{} ... \n".format(epoch + 1, num_epochs),
              "Train Loss: {:.3f} \n".format(running_loss),
              "Test Loss: {:.3f} \n".format(test_running_loss),
              "Train Accuracy: {:.3f} \n".format(accuracy),
              "Test Accuracy: {:.3f} \n".format(test_accuracy),
              "Time: {:.2f} m".format((time.time() - since) / 60))
        
    history = {'train_loss' : train_losses, 'test_loss': test_losses,
               'train_acc': train_accs, 'val_acc': test_accs,
               'lrs': lrs}
    print('Total time: {:.2f} m' .format((time.time()- start_time) / 60))
    return history

In [ ]:
history = train(model, 
          train_data_loader, 
          train_data_loader, 
          criterion, 
          optimizer, 
          lr_scheduler,
          NUM_EPOCHS,
          device)

## Extract emebeddings of the training set

In [ ]:
# replace the fully connected layer with an identity layer
model.fc = nn.Identity()
model.to(device)
model.eval()

In [ ]:
import time
start = time.time()
embeddings = []
class_ids = []
for data in train_data_loader:
    image_tensors, class_id_tensors = data
    image_tensors = image_tensors.to(device)
    # class_id_tensors = class_id_tensors.to(device)    
   
    with torch.no_grad():
        features = model(image_tensors)
    embeddings.append(features.cpu())
    class_ids.append(class_id_tensors)
# create a tensor
embeddings = torch.cat(embeddings, dim = 0)
class_ids = torch.cat(class_ids, dim = 0)
# convert to numpy before any dimentionality reduction
embeddings = embeddings.numpy()
class_ids = class_ids.numpy()
elap = time.time() - start

## T-SNE visualization

In [ ]:
from sklearn.manifold import TSNE
import seaborn as sn
# we want to get T-SNE embedding with 2 dimensions
n_components = 2
tsne = TSNE(n_components)
tsne_result = tsne.fit_transform(embeddings)

In [ ]:
# Plot the result of our TSNE with the label color coded
# A lot of the stuff here is about making the plot look pretty and not TSNE
# pick the same number of samples from each class for display (to make the distribution more clear)
unique_class_ids = np.unique(class_ids)
idxs_per_class_id = {}

MIN_NUM_REQUIRED_SAMPLES = 1000
MAX_NUM_REQUIRED_SAMPLES = 3000
num_samples = 1e9

for class_id in unique_class_ids:
    idxs = np.where(class_ids == class_id)[0]
    if len(idxs) == 0:
        continue
    np.random.shuffle(idxs)
    idxs_per_class_id[class_id] = idxs
    
    if len(idxs) < MIN_NUM_REQUIRED_SAMPLES:
        continue
    num_samples = min(num_samples, len(idxs))

num_samples = min(num_samples, MAX_NUM_REQUIRED_SAMPLES)

idxs_to_use = np.hstack([c_idxs[:num_samples] for _, c_idxs in idxs_per_class_id.items()])

tsne_result_df = pd.DataFrame({'tsne_1': tsne_result[idxs_to_use, 0], 'tsne_2': tsne_result[idxs_to_use, 1], 'label': class_ids[idxs_to_use]})
tsne_result_df['label'] = tsne_result_df['label'].map(CLASSIFIER_LABEL_MAP)
fig, ax = plt.subplots(1, figsize=(10, 10))
sn_plot = sn.scatterplot(x='tsne_1', y='tsne_2', hue='label', data=tsne_result_df, ax=ax,s=15)
lim = (tsne_result.min()-5, tsne_result.max()+5)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(0.17, 0.8), loc=0, borderaxespad=0.0)

In [ ]:
sn_plot.get_figure().savefig("cellanome_cells_figures/cellanome_adhered_cell_clusters_t_sne_dinov2_l.png")

## UMAP visualization

In [ ]:
import umap
# UMAP embeddings will be with 2 dimensions
reducer = umap.UMAP()
umap_result = reducer.fit_transform(embeddings)

In [ ]:
# Plot the result of our UMAP with the label color coded
unique_class_ids = np.unique(class_ids)
idxs_per_class_id = {}

MIN_NUM_REQUIRED_SAMPLES = 1000
MAX_NUM_REQUIRED_SAMPLES = 2500
num_samples = 1e9

for class_id in unique_class_ids:
    idxs = np.where(class_ids == class_id)[0]
    if len(idxs) == 0:
        continue
    np.random.shuffle(idxs)
    idxs_per_class_id[class_id] = idxs
    
    if len(idxs) < MIN_NUM_REQUIRED_SAMPLES:
        continue
    num_samples = min(num_samples, len(idxs))

num_samples = min(num_samples, MAX_NUM_REQUIRED_SAMPLES)

idxs_to_use = np.hstack([c_idxs[:num_samples] for _, c_idxs in idxs_per_class_id.items()])

umap_result_df = pd.DataFrame({'umap_1': umap_result[idxs_to_use, 0], 'umap_2': umap_result[idxs_to_use, 1], 'label': class_ids[idxs_to_use]})
umap_result_df['label'] = umap_result_df['label'].map(CLASSIFIER_LABEL_MAP)
fig, ax = plt.subplots(1, figsize=(10, 10))
sn_plot = sn.scatterplot(x='umap_1', y='umap_2', hue='label', data=umap_result_df, ax=ax,s=10)
lim_x = (umap_result[idxs_to_use, 0].min()-1, umap_result[idxs_to_use, 0].max()+1)
lim_y = (umap_result[idxs_to_use, 1].min()-1, umap_result[idxs_to_use, 1].max()+1)
ax.set_xlim(lim_x)
ax.set_ylim(lim_y)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(0.9, 0.95), loc=1, borderaxespad=0.0)

In [ ]:
sn_plot.get_figure().savefig("cellanome_cells_figures/cellanome_adhered_cell_clusters_umap_dinov2_l.png")

In [ ]:
# here, we are trying to calculated an average embedding for all cell crops belonging to the same cell type
# we use this to compute a distance between cell types based on this average embedding
cell_type_embeddings = np.zeros((len(CLASSIFIER_LABEL_MAP), embeddings.shape[1]))
cell_type_var = np.zeros((len(CLASSIFIER_LABEL_MAP), embeddings.shape[1]))
for class_id in CLASSIFIER_LABEL_MAP:
    idxs: np.ndarray = np.where(class_ids == class_id)[0]
    cell_type_embeddings[class_id, :] = np.mean(embeddings[idxs, :], axis = 0)
    cell_type_var[class_id, :] = np.var(embeddings[idxs, :], axis = 0)

In [ ]:
dist = np.zeros((len(CLASSIFIER_LABEL_MAP), len(CLASSIFIER_LABEL_MAP)))
similar_types = {}
for class_id_i in CLASSIFIER_LABEL_MAP:
    for class_id_j in CLASSIFIER_LABEL_MAP:
        # Euclidean distance
        dist[class_id_i, class_id_j] = (np.sum((cell_type_embeddings[class_id_i] - 
                                                cell_type_embeddings[class_id_j]) ** 2) ** 0.5) / (
            np.sum(cell_type_var[class_id_i] ** 2) + np.sum(cell_type_var[class_id_j] ** 2)
        ) ** 0.5
        # dist[class_id_i, class_id_j] = (np.sum((cell_type_embeddings[class_id_i] - 
        #                                         cell_type_embeddings[class_id_j]) ** 2) ** 0.5) / (
        #     np.sum(cell_type_var[class_id_j] ** 2)
        # ) ** 0.5
        # dist[class_id_i, class_id_j] = np.sum((cell_type_embeddings[class_id_i] - 
        #                                        cell_type_embeddings[class_id_j]) ** 2) ** 0.5
        # cosine similarity
        # dist[class_id_i, class_id_j] = np.sum(cell_type_embeddings[class_id_i] * cell_type_embeddings[class_id_j])
        # dist[class_id_i, class_id_j] /= np.sqrt(np.sum(cell_type_embeddings[class_id_i] ** 2))
        # dist[class_id_i, class_id_j] /= np.sqrt(np.sum(cell_type_embeddings[class_id_j] ** 2))
    sorted_idxs = np.argsort(dist[class_id_i])
    similar_types[CLASSIFIER_LABEL_MAP[class_id_i]] = [CLASSIFIER_LABEL_MAP[i] for i in sorted_idxs if i != class_id_i]

In [ ]:
import seaborn as sn
df_distance = pd.DataFrame(dist, index = list(CLASSIFIER_LABEL_MAP.values()),
                     columns = list(CLASSIFIER_LABEL_MAP.values()))
plt.figure(figsize = (12,7))
sn_plot = sn.heatmap(df_distance, annot=True)

In [ ]:
sn_plot.get_figure().savefig("cellanome_cells_figures/cellanome_adhered_cell_distances.png")

In [ ]:
similar_types

In [ ]:
var = np.sum(cell_type_var ** 2, axis=1) ** 0.5

In [ ]:
var

In [ ]:
f1_score = np.array(
    [[0.873,	0.264,	0.382,	0.511,	0.312,	0.005],
     [0.785,	0.738,	0.692,	0.700,	0.371,	0.006],
     [0.708,	0.607,	0.847,	0.736,	0.571,	0.047],
     [0.782,	0.510,	0.561,	0.768,	0.613,	0.021],
     [0.530,	0.277,	0.729,	0.637,	0.813,	0.090],
     [0.047,	0.112,	0.480,	0.408,	0.593,	0.852]])

In [ ]:
f1_score_box = np.array(
[[0.914, 	0.543, 	0.461, 	0.636, 	0.361, 	0.004],
 [0.804, 	0.846, 	0.708, 	0.732,	0.409, 	0.004],
 [0.689,	0.659, 	0.857, 	0.741, 	0.608, 	0.029],
 [0.815, 	0.636, 	0.583, 	0.802, 	0.664, 	0.013],
 [0.524,	0.368, 	0.757, 	0.695, 	0.823, 	0.061],
 [0.030, 	0.057, 	0.410, 	0.226, 	0.478, 	0.859]])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# create data
for class_id in CLASSIFIER_LABEL_MAP:
    sorted_idxs = np.argsort(dist[class_id])
    x = dist[class_id, sorted_idxs[1:]]
    y = f1_score[class_id, sorted_idxs[1:]]
    plt.plot(x, y, label = CLASSIFIER_LABEL_MAP[class_id], linestyle="--", marker='o')
    count = 1
    for i, j in zip(x, y):
        plt.text(i, j, CLASSIFIER_LABEL_MAP[sorted_idxs[count]])
        count+=1
plt.xlabel("Distance from the cell type used in training")
plt.ylabel("F1 score")
plt.legend()
plt.show()



In [ ]:
c = ['r', 'g', 'b', 'c', 'y', 'm']
fig, ax = plt.subplots(2, 3, figsize=(15, 15))
# create data
for class_id in CLASSIFIER_LABEL_MAP:
    sorted_idxs = np.argsort(dist[class_id])
    x = dist[class_id, sorted_idxs[1:]]
    y = f1_score[class_id, sorted_idxs[1:]]
    row_idx = class_id // 3
    col_idx = class_id % 3
    ax[row_idx, col_idx].plot(x, y, label = CLASSIFIER_LABEL_MAP[class_id], linestyle="--", marker='o', color=c[class_id])
    count = 1
    for i, j in zip(x, y):
        ax[row_idx, col_idx].text(i, j, CLASSIFIER_LABEL_MAP[sorted_idxs[count]])
        count+=1
    ax[row_idx, col_idx].set_title(f"trained on {CLASSIFIER_LABEL_MAP[class_id]}") 
    ax[row_idx, col_idx].set(ylabel='F1 score', xlabel='Distance')
# plt.xlabel("Distance from the cell type used in training")
# plt.ylabel("F1 score")
# plt.legend()
plt.savefig("cellanome_cells_figures/scatter_plots.png")
plt.show()

In [ ]:
"""
Adhered cells

IMR-90: Except for Hs675T Fibroblasts that are expected to have high F1 score because of similarity to IMR-90 cells, the rest makes sense. 
Hs675T Fibroblasts: All makes sense. 
HeLa: mutuDC is expected to have a better F1 score than IMR-90 and Hs675T, but the rest are in the right order. 
MC38: HeLa is expected to have worse F1 score. The rest are OK. 
mutuDC: Almost OK
Neuron: All makes sense

- Euclidean distance (normalized by the sum of variances or the target variance - both leads to the same results)
 'IMR90': ['Hs675T', 'MC38', 'HeLa', 'mutuDC', 'Neuron'],
 'Hs675T': ['IMR90', 'MC38', 'HeLa', 'mutuDC', 'Neuron'],
 'HeLa': ['MC38', 'mutuDC', 'IMR90', 'Hs675T', 'Neuron'],
 'MC38': ['HeLa', 'IMR90', 'mutuDC', 'Hs675T', 'Neuron'],
 'mutuDC': ['MC38', 'HeLa', 'IMR90', 'Hs675T', 'Neuron'],
 'Neuron': ['MC38', 'mutuDC', 'HeLa', 'IMR90', 'Hs675T']

- Euclidean distance (no normalization)
 'IMR90': ['Hs675T', 'MC38', 'mutuDC', 'HeLa', 'Neuron'],
 'Hs675T': ['IMR90', 'MC38', 'HeLa', 'mutuDC', 'Neuron'],
 'HeLa': ['MC38', 'mutuDC', 'IMR90', 'Hs675T', 'Neuron'],
 'MC38': ['HeLa', 'mutuDC', 'IMR90', 'Hs675T', 'Neuron'],
 'mutuDC': ['MC38', 'HeLa', 'IMR90', 'Hs675T', 'Neuron'],
 'Neuron': ['mutuDC', 'MC38', 'HeLa', 'IMR90', 'Hs675T']

Suspension cells


- Euclidean distance (normalized by the sum target variance)
 'Jurkat': ['Raji', 'NK92', 'TALL104', 'PBMC', 'K562', 'HeLa', 'IMR90'],
 'K562': ['IMR90', 'NK92', 'Jurkat', 'Raji', 'TALL104', 'HeLa', 'PBMC'],
 'Raji': ['Jurkat', 'NK92', 'TALL104', 'K562', 'HeLa', 'PBMC', 'IMR90'],
 'HeLa': ['Raji', 'Jurkat', 'K562', 'NK92', 'TALL104', 'IMR90', 'PBMC'],
 'NK92': ['K562', 'Jurkat', 'TALL104', 'Raji', 'IMR90', 'PBMC', 'HeLa'],
 'IMR90': ['K562', 'NK92', 'TALL104', 'Jurkat', 'Raji', 'PBMC', 'HeLa'],
 'PBMC': ['Jurkat', 'TALL104', 'NK92', 'Raji', 'K562', 'IMR90', 'HeLa'],
 'TALL104': ['NK92', 'Jurkat', 'PBMC', 'Raji', 'K562', 'IMR90', 'HeLa']


"""

| Training datasets | adhered_imr90	| Hs675Tfibroblasts	| adhered_hela | mc38 | mutuDC| adhered_neuron |
| --- | --- | --- | --- | --- | --- | --- |
| adhered_imr90	| 0.873	| 0.264	| 0.382 | 0.511 |	0.312 | 0.005 |
| Hs675Tfibroblasts	| 0.785 | 0.738 | 0.692	| 0.700 | 0.371 | 0.006 |
| adhered_hela	| 0.708 | 0.607 | 0.847 | 0.736 | 0.571 | 0.047 |
| mc38 | 0.782 | 0.510 | 0.561 | 0.768 | 0.613 | 0.021 |
| mutuDC | 0.530 | 0.277 |	0.729 | 0.637	| 0.813	| 0.090 |
| adhered_neuron	| 0.047	| 0.112	| 0.480 | 0.408 | 0.593 | 0.852 |


| Training datasets | Jurkat | K562 | HeLa | Raji |	NK92 |IMR90 | Tall104 |	PBMC |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Jurkat |	0.9706 | 0.9615 | 0.9794 | 0.9659	| 0.9017 |	0.8581 | 0.7124	| 0.7098 |
| NK92	| 0.9598 | 0.9626 |	0.9781 | 0.9455	| 0.9116 | 0.8603 |	0.6701 | 0.6532 |
| IMR90	| 0.9310 | 0.9615	| 0.9806 |	0.9097	| 0.8944 |	0.8963 | 0.6421	| 0.5732 |
| Tall104 |	0.9585 | 0.9504	| 0.9736 | 0.9751 | 0.8612 | 0.8383	| 0.8345 | 0.6661 |
| PBMC	| 0.9701 | 0.9536	| 0.9787 |	0.9399	| 0.8863 |	0.8315 | 0.7011	| 0.9617 |